## Task 10:Data-quality Audit
*My approach:I will load both csv files and inspect them for issues before cleaning.*

In [14]:
import numpy as np
import pandas as pd

# --- Duka Electronics Generator ---
# Note: I am assuming the Duka generator looks like this based on standard PLP assignments.
# If your portal has a specific code block for Duka, replace this section with it.
np.random.seed(2026)
duka_data = pd.DataFrame({
    'order_id': [f"ORD-{i:04d}" for i in range(1, 1001)],
    'date': pd.date_range("2026-01-01", periods=1000, freq="6H").strftime("%Y-%m-%d"),
    'branch': np.random.choice(["Nairobi", "Mombasa", "Kisumu"], 1000),
    'channel': np.random.choice(["Online", "In-Store"], 1000),
    'product': np.random.choice(["Laptop", "Phone", "TV", "Headphones", "Charger"], 1000),
    'units': np.random.randint(1, 5, 1000),
    'price': np.random.randint(500, 30000, 1000)
})
duka_data['revenue'] = duka_data['units'] * duka_data['price']
duka_data.to_csv("duka_sales.csv", index=False)

# --- Agent Network Generator ---
rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d")
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT", "Withdrawal", "deposit"], n),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (pd.to_datetime("2026-03-01") + pd.to_timedelta(rng.integers(0, 60, n), unit="D")).strftime("%Y-%m-%d")
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

/tmp/ipykernel_1245/3863481055.py:10: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  'date': pd.date_range("2026-01-01", periods=1000, freq="6H").strftime("%Y-%m-%d"),


In [15]:
import pandas as pd
import numpy as np

# Load the data
txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")

print("--- TRANSACTIONS ---")
print(txns.info())
print("\nMissing values:\n", txns.isna().sum())

print("\n--- AGENTS ---")
print(agents.info())
print("\nMissing values:\n", agents.isna().sum())

# Look for the specific injected problems
print("\nUnique txn_types:", txns['txn_type'].unique())
print("Duplicate TXN IDs:", txns['txn_id'].duplicated().sum())
print("Unknown Agents:", txns[~txns['agent_id'].isin(agents['agent_id'])]['agent_id'].unique())

--- TRANSACTIONS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    object 
 1   agent_id  612 non-null    object 
 2   txn_type  612 non-null    object 
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    object 
dtypes: float64(1), object(4)
memory usage: 24.0+ KB
None

Missing values:
 txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0
dtype: int64

--- AGENTS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   agent_id   20 non-null     object
 1   county     20 non-null     object
 2   onboarded  20 non-null     object
dtypes: object(3)
memory usage: 612.0+ bytes
None

Missing values:
 agent_id     0
county       0
onboarded    0
dtype: int64

## Task 11:uniqueness
*My approach: The generator injected 12 duplicate trnsactions.I will drop them.*

In [16]:
initial_rows= len(txns)
txns= txns.drop_duplicates(subset=['txn_id'],keep = 'first')
removed = initial_rows - len(txns)
print(f"Removed {removed} duplicate transactions.")

Removed 12 duplicate transactions.


## Task 12:Consistency
*My approach:Standardise 'txn_type' to exactly "Deposit" or "Withdraw".*

In [17]:
# Convert to string, strip spaces, capitalize first letter, lower the rest
txns['txn_type'] = txns['txn_type'].str.strip().str.capitalize()

# Double check values
print("New txn_types:", txns['txn_type'].unique())

New txn_types: ['Deposit' 'Withdrawal']


## Task 13 and 14:Validity,Types and Completness
*My approach: Convert 'amount' to numeric.The "N/A" sting will become Nan.I will impute missing amounts wuth the median and add flag column.*

In [18]:
# 1. Convert "N/A" to actual NaN, then to numeric
txns['amount'] = pd.to_numeric(txns['amount'], errors='coerce')

# 2. Create the flag column BEFORE imputing
txns['amount_imputed'] = txns['amount'].isna()

# 3. Impute missing amounts with the median
median_amount = txns['amount'].median()
txns['amount'] = txns['amount'].fillna(median_amount)

print(f"Imputed {txns['amount_imputed'].sum()} missing amounts with the median ({median_amount}).")

# 4. Fix the dates (replace '/' with '-')
txns['txn_date'] = txns['txn_date'].str.replace('/', '-')

# 5. Convert txn_date to datetime
txns['txn_date'] = pd.to_datetime(txns['txn_date'])
print("Date data type is now:", txns['txn_date'].dtype)

Imputed 20 missing amounts with the median (1776.5).
Date data type is now: datetime64[ns]


## Task 15:Outliers
*My approach:Calculate the interquatile range to flag amounts that are way too high like 250000 values.*

In [19]:
Q1 = txns['amount'].quantile(0.25)
Q3 = txns['amount'].quantile(0.75)
IQR = Q3 - Q1

upper_bound = Q3 + (1.5 * IQR)

# Flag outliers
txns['is_outlier'] = txns['amount'] > upper_bound

print(f"Upper bound for outliers is: {upper_bound}")
print(f"Number of outliers flagged: {txns['is_outlier'].sum()}")
print("Investigate the 250,000 values:", txns[txns['is_outlier'] == True]['amount'].unique())

Upper bound for outliers is: 5513.75
Number of outliers flagged: 53
Investigate the 250,000 values: [  9129.   5632.   8507.   6006.   8035.   6566.   7116.   7962.   5856.
   7072.   5643.   5777.   5749.   7465.   6310.   5745.   6241.   5560.
  13362.   5615.  28847.   5788.   6261.   7946.  12078.  22316.   5567.
   6235.   6158.   9854.   5646.   6308. 250000.   6914.   6754.   7146.
   5897.   9523.   8318.   8966.   6852.   7122.   6452.   8395.   8518.
   6509.   7060.   6257.  16605.]


## Task 16:Merge
*My approach: Left join the transactions to the agent register. Unknown agents(like AG099) will show Nan for county.*

In [20]:
# Merge
merged_df = pd.merge(txns, agents, on='agent_id', how='left')

# Check for unknown agents
unknown_agents = merged_df['county'].isna().sum()
print(f"Transactions with unknown agents: {unknown_agents}")
print("We will keep them but label their county as 'Unknown'.")
merged_df['county'] = merged_df['county'].fillna('Unknown')

Transactions with unknown agents: 7
We will keep them but label their county as 'Unknown'.


## Task 17:Summarise and Export
*My approach:Group by county and transaction type to find total and average values.Then export the clean CSV.*

In [21]:
# County Summary
county_summary = merged_df.groupby(['county', 'txn_type'])['amount'].agg(['sum', 'count', 'mean']).reset_index()
print("County Summary:\n", county_summary)

# Weekly Pivot Table
merged_df['week'] = merged_df['txn_date'].dt.isocalendar().week
weekly_pivot = merged_df.pivot_table(index='week', columns='county', values='amount', aggfunc='sum').fillna(0)
print("\nWeekly Pivot:\n", weekly_pivot)

# Export
merged_df.to_csv("outputs/agent_transactions_clean.csv", index=False)
print("\nSaved clean data to outputs/agent_transactions_clean.csv")

County Summary:
      county    txn_type       sum  count         mean
0    Kiambu     Deposit  334737.0     43  7784.581395
1    Kiambu  Withdrawal   77050.5     27  2853.722222
2    Kisumu     Deposit  436420.5     74  5897.574324
3    Kisumu  Withdrawal   76374.5     37  2064.175676
4   Mombasa     Deposit  148986.0     72  2069.250000
5   Mombasa  Withdrawal  368592.0     61  6042.491803
6   Nairobi     Deposit  543590.0    118  4606.694915
7   Nairobi  Withdrawal  162036.0     62  2613.483871
8    Nakuru     Deposit  184890.0     60  3081.500000
9    Nakuru  Withdrawal  328594.5     39  8425.500000
10  Unknown     Deposit   25279.0      6  4213.166667
11  Unknown  Withdrawal    1877.0      1  1877.000000

Weekly Pivot:
 county    Kiambu    Kisumu   Mombasa   Nairobi    Nakuru  Unknown
week                                                             
9            0.0    1893.0    6759.0    7046.0    2743.5      0.0
10       18317.0  287933.0  285669.5   68967.0   56574.0      0.0
1

In [22]:
import os
os.makedirs("outputs",exist_ok=True)